In [1]:
!pip install faker mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   -------------------------- ------------- 1.3/2.0 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.1 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


### Synthetic Customer Dataset

In [1]:
import pandas as pd 
import numpy as np 
from faker import Faker 
fake = Faker()

num_customers = 10000

customers = pd.DataFrame({
    "Customer_id":range(1,num_customers+1),
    "Age":np.random.randint(21,65,num_customers),
    "Income": np.random.randint(20000, 150000, num_customers),
    "Employment_type": np.random.choice(
        ["Salaried", "Self-employed", "Business", "Student"],
        num_customers
    ),
    "City": [fake.city() for _ in range(num_customers)],
    "Credit_score": np.random.randint(300, 850, num_customers)
})
customers.sample()

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


,Customer_id,Age,Income,Employment_type,City,Credit_score
9838,9839,40,27265,Salaried,Danielport,472


### Synthetic Transaction Dataset

In [2]:
num_transactions = 50000

transactions = pd.DataFrame({
    "transaction_id": range(1, num_transactions + 1),
    "customer_id": np.random.randint(1, num_customers + 1, num_transactions),
    "transaction_date": pd.to_datetime(
        np.random.choice(pd.date_range("2023-01-01", "2024-12-31"), num_transactions)
    ),
    "amount": np.round(np.random.uniform(5, 2000, num_transactions), 2),
    "category": np.random.choice(
        ["Groceries", "Shopping", "Travel", "Food", "Entertainment", "Bills"],
        num_transactions
    ),
    "merchant": [fake.company() for _ in range(num_transactions)]
})

transactions.head()

,transaction_id,customer_id,transaction_date,amount,category,merchant
0,1,4580,2024-10-27,699.56,Travel,Haley Ltd
1,2,7469,2023-04-12,1016.87,Shopping,Frazier and Sons
2,3,3910,2024-06-29,208.19,Shopping,Gibson-Dunn
3,4,1499,2024-11-18,1547.67,Groceries,"Miller, Smith and Ortiz"
4,5,7767,2023-01-03,820.25,Bills,Jones and Sons


In [10]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date']).dt.to_pydatetime()

### Create Loan Dataset (Risk Data)

In [3]:
num_loans = 5000

loans = pd.DataFrame({
    "Loan_id": range(1, num_loans + 1),
    "Customer_id": np.random.randint(1, num_customers + 1, num_loans),
    "Loan_amount": np.random.randint(5000, 50000, num_loans),
    "Interest_rate": np.round(np.random.uniform(5, 18, num_loans), 2),
})

loans["EMI"] = loans["Loan_amount"] * loans["Interest_rate"] / 100 / 12

loans["Default_flag"] = np.random.choice(
    [0,1],
    num_loans,
    p=[0.8,0.2]  # 20% defaults
)

loans.head()

,Loan_id,Customer_id,Loan_amount,Interest_rate,EMI,Default_flag
0,1,895,38596,13.63,438.386233,0
1,2,6729,46385,5.21,201.388208,0
2,3,115,16148,16.23,218.401700,1
3,4,3119,49291,11.63,477.711942,1
4,5,7899,8709,12.41,90.065575,0


## Connecting python to MySQL

In [4]:
import mysql.connector 

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="*********",
    database="financial_risk_system"
)
cursor = conn.cursor()    # cursor is the object that sends SQL commands to the database. think of it like this: 
# Python program
#     ↓
# Cursor (communicates)
#     ↓
# MySQL Database

## Insert customer into MySQL 

In [12]:
cursor.executemany(
    "INSERT IGNORE INTO customers VALUES (%s,%s,%s,%s,%s,%s)",
    customers.values.tolist()
)

conn.commit()

## Insert transactions  into MySQL 

In [13]:
cursor.executemany(
    "INSERT IGNORE INTO transactions VALUES (%s,%s,%s,%s,%s,%s)",
    transactions.values.tolist()
)

conn.commit()

## Insert loans into MySQL 

In [14]:
cursor.executemany(
    "INSERT IGNORE INTO loans VALUES (%s,%s,%s,%s,%s,%s)",
    loans.values.tolist()
)

conn.commit()